In [ ]:
import os
import json
import copy
import random
import warnings
from typing import List, Dict, Tuple, Optional

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore", message=".*VisibleDeprecationWarning.*")
warnings.filterwarnings("ignore", message=".*dtype\\(\\): align should be passed.*")

QUICK_RUN = False
SAVE_PATH = "./outputs/kfn_split_cifar100_improved.json"
REPLAY_BUFFER_CAPACITY = 2000  # Increased for proper 100-way Bias Correction
N_TASKS = 10
CLASSES_PER_TASK = 10
TOTAL_CLASSES = N_TASKS * CLASSES_PER_TASK
EPOCHS_BASE = 200 if not QUICK_RUN else 5  # Increased to harden the foundational Task 1 filters
EPOCHS_SPECIALIST = 50 if not QUICK_RUN else 3
EPOCHS_FUSION = 80 if not QUICK_RUN else 5
EPOCHS_BIAS = 15 if not QUICK_RUN else 3
BATCH_SIZE = 128
NOVELTY_LOW = 0.15
NOVELTY_MID = 0.30
KD_TEMPERATURE = 2.0
USE_AMP = True
FORCE_CPU = False
EMA_MOMENTUM = 0.999
NUM_WORKERS = 2


def set_deterministic(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def get_device(force_cpu: bool = False) -> torch.device:
    if force_cpu or not torch.cuda.is_available():
        return torch.device("cpu")
    return torch.device("cuda")


def save_progress(obj: Dict, path: str = SAVE_PATH) -> None:
    directory = os.path.dirname(path)
    if directory:
        os.makedirs(directory, exist_ok=True)
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)


class CIFAR100TaskDataset(Dataset):
    def __init__(self, dataset: Dataset, classes: List[int]):
        self.dataset = dataset
        self.classes = classes
        self.indices = [i for i, label in enumerate(dataset.targets) if label in classes]

    def __len__(self) -> int:
        return len(self.indices)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        img, label = self.dataset[self.indices[idx]]
        return img, label


class TensorDatasetWrapper(Dataset):
    def __init__(self, images: torch.Tensor, labels: torch.Tensor):
        self.images = images
        self.labels = labels

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, int]:
        return self.images[idx], self.labels[idx]


def get_task_classes(task_id: int) -> List[int]:
    start_class = (task_id - 1) * CLASSES_PER_TASK
    end_class = task_id * CLASSES_PER_TASK
    return list(range(start_class, end_class))


def get_task_loaders(
    train_ds: Dataset,
    test_ds: Dataset,
    task_id: int,
    batch_size: int = BATCH_SIZE,
    num_workers: int = NUM_WORKERS,
) -> Tuple[DataLoader, DataLoader]:
    task_classes = get_task_classes(task_id)
    train_subset = CIFAR100TaskDataset(train_ds, task_classes)
    test_subset = CIFAR100TaskDataset(test_ds, task_classes)
    pin_memory = torch.cuda.is_available() and not FORCE_CPU

    train_loader = DataLoader(
        train_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    test_loader = DataLoader(
        test_subset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )
    return train_loader, test_loader


class DynamicReplayBuffer:
    def __init__(self, capacity_per_class: int = REPLAY_BUFFER_CAPACITY):
        self.capacity_per_class = capacity_per_class
        self.storage: Dict[int, List[Tuple[torch.Tensor, float]]] = {}

    def add_exemplars_from_dataset(
        self,
        dataset: Dataset,
        classes: List[int],
        feature_extractor: nn.Module,
        device: torch.device,
        batch_size: int = BATCH_SIZE,
        num_workers: int = NUM_WORKERS,
    ) -> None:
        pin_memory = torch.cuda.is_available() and not FORCE_CPU
        loader = DataLoader(
            CIFAR100TaskDataset(dataset, classes),
            batch_size=batch_size,
            shuffle=False,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

        by_class_images: Dict[int, List[torch.Tensor]] = {c: [] for c in classes}
        by_class_feats: Dict[int, List[torch.Tensor]] = {c: [] for c in classes}

        feature_extractor.eval()
        with torch.no_grad():
            for x, y in loader:
                x = x.to(device)
                feats = feature_extractor(x)
                feats = F.normalize(F.adaptive_avg_pool2d(feats, 1).flatten(1), p=2, dim=1).cpu()
                for img, label, feat in zip(x.cpu(), y.cpu(), feats):
                    label_i = int(label.item())
                    by_class_images[label_i].append(img)
                    by_class_feats[label_i].append(feat)

        for cls in classes:
            if len(by_class_images[cls]) == 0:
                continue
            feats = torch.stack(by_class_feats[cls])
            centroid = F.normalize(feats.mean(dim=0, keepdim=True), p=2, dim=1)
            scores = torch.sum(feats * centroid, dim=1)
            k = min(self.capacity_per_class, scores.numel())
            topk = torch.topk(scores, k=k, largest=True).indices.tolist()
            self.storage[cls] = [(by_class_images[cls][idx], float(scores[idx].item())) for idx in topk]

    def get_loader(
        self,
        batch_size: int = BATCH_SIZE,
        num_workers: int = NUM_WORKERS,
        shuffle: bool = True,
    ) -> Optional[DataLoader]:
        if not self.storage:
            return None

        images = []
        labels = []
        for cls in sorted(self.storage.keys()):
            for img, _ in self.storage[cls]:
                images.append(img)
                labels.append(cls)

        if not images:
            return None

        images_t = torch.stack(images)
        labels_t = torch.tensor(labels, dtype=torch.long)
        pin_memory = torch.cuda.is_available() and not FORCE_CPU
        return DataLoader(
            TensorDatasetWrapper(images_t, labels_t),
            batch_size=batch_size,
            shuffle=shuffle,
            num_workers=num_workers,
            pin_memory=pin_memory,
        )

    def __len__(self) -> int:
        return sum(len(v) for v in self.storage.values())


class CosineLinear(nn.Module):
    def __init__(self, in_features: int, out_features: int, sigma: float = 16.0):
        super().__init__()
        self.weight = nn.Parameter(torch.Tensor(out_features, in_features))
        self.sigma = nn.Parameter(torch.tensor([sigma], dtype=torch.float32))
        self.reset_parameters()

    def reset_parameters(self) -> None:
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.normalize(x, p=2, dim=1)
        w = F.normalize(self.weight, p=2, dim=1)
        return self.sigma * F.linear(x, w)


class SupConLoss(nn.Module):
    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, features: torch.Tensor, labels: torch.Tensor) -> torch.Tensor:
        device = features.device
        batch_size = features.shape[0]
        labels = labels.contiguous().view(-1, 1)
        mask = torch.eq(labels, labels.T).float().to(device)

        anchor_dot_contrast = torch.div(torch.matmul(features, features.T), self.temperature)
        logits_max, _ = torch.max(anchor_dot_contrast, dim=1, keepdim=True)
        logits = anchor_dot_contrast - logits_max.detach()

        logits_mask = torch.scatter(
            torch.ones_like(mask),
            1,
            torch.arange(batch_size).view(-1, 1).to(device),
            0,
        )
        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(1, keepdim=True) + 1e-8)
        mean_log_prob_pos = (mask * log_prob).sum(1) / (mask.sum(1) + 1e-8)
        return -mean_log_prob_pos.mean()


def linear_cka(f1: torch.Tensor, f2: torch.Tensor) -> torch.Tensor:
    """Computes the Linear Centered Kernel Alignment (CKA) to measure feature redundancy."""
    f1_c = f1 - f1.mean(dim=0)
    f2_c = f2 - f2.mean(dim=0)

    g1 = torch.mm(f1_c, f1_c.t())
    g2 = torch.mm(f2_c, f2_c.t())

    hsic = torch.sum(g1 * g2)
    var1 = torch.sqrt(torch.sum(g1 * g1))
    var2 = torch.sqrt(torch.sum(g2 * g2))

    return hsic / (var1 * var2 + 1e-8)


class Specialist(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = torchvision.models.resnet18(weights=None)
        resnet.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()
        self.features = nn.Sequential(*list(resnet.children())[:-2])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.features(x)


class StructuralFusionModule(nn.Module):
    def __init__(self, in_channels: int, old_dim: int, new_dim: int, novelty_score: float):
        super().__init__()
        self.novelty_score = novelty_score
        self.has_expansion = new_dim > 0

        self.proj_reuse = nn.Sequential(
            nn.Conv2d(in_channels, old_dim, 1, bias=False),
            nn.BatchNorm2d(old_dim),
            nn.ReLU(inplace=True),
            nn.Conv2d(old_dim, old_dim, 3, padding=1, bias=False),
            nn.BatchNorm2d(old_dim),
            nn.ReLU(inplace=True),
        )
        self.gate_reuse = nn.Parameter(torch.tensor([0.0]))

        if self.has_expansion:
            self.proj_expand = nn.Sequential(
                nn.Conv2d(in_channels, new_dim, 1, bias=False),
                nn.BatchNorm2d(new_dim),
                nn.ReLU(inplace=True),
                nn.Conv2d(new_dim, new_dim, 3, padding=1, bias=False),
                nn.BatchNorm2d(new_dim),
                nn.ReLU(inplace=True),
            )

    def forward(self, x: torch.Tensor, old_memory_detached: torch.Tensor) -> Tuple[torch.Tensor, Optional[torch.Tensor]]:
        reuse_scale = max(0.2, 0.7 - self.novelty_score * 0.4)
        delta_old = self.proj_reuse(x) * torch.sigmoid(self.gate_reuse) * reuse_scale
        feat_new = None
        if self.has_expansion:
            feat_new = self.proj_expand(x)
            feat_new = feat_new - feat_new.mean(dim=(2, 3), keepdim=True)
        return delta_old, feat_new


class GlobalModel(nn.Module):
    def __init__(
        self,
        n_specs: int,
        ch: int,
        n_classes: int,
        old_dim: int = 256,
        new_dim: int = 0,
        novelty_score: float = 0.0,
        use_weight_align: bool = True,
    ):
        super().__init__()
        self.old_dim = old_dim
        self.new_dim = new_dim
        self.use_weight_align = use_weight_align

        self.old_proj = nn.ModuleList([nn.Conv2d(ch, ch, 1, bias=False) for _ in range(n_specs)])
        self.spec_attention = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, max(ch // 8, 1), 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(max(ch // 8, 1), 1, 1),
        )
        self.old_gate = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Conv2d(ch, ch // 4, 1),
            nn.ReLU(inplace=True),
            nn.Conv2d(ch // 4, ch, 1),
            nn.Sigmoid(),
        )
        self.old_bottleneck = nn.Sequential(
            nn.Conv2d(ch, old_dim, 1, bias=False),
            nn.BatchNorm2d(old_dim),
            nn.ReLU(inplace=True),
        )
        self.fusion = StructuralFusionModule(ch, old_dim, new_dim, novelty_score) if new_dim > 0 else None
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = CosineLinear(old_dim + new_dim, n_classes, sigma=16.0)

    def forward_old(self, feats: List[torch.Tensor]) -> torch.Tensor:
        projected = [proj(feat) for proj, feat in zip(self.old_proj, feats)]
        gated = [p * self.old_gate(p) for p in projected]
        scores = [self.spec_attention(g).flatten(1) for g in gated]
        score_tensor = torch.stack(scores, dim=1)
        weights = torch.softmax(score_tensor, dim=1)
        fused = 0.0
        for i, g in enumerate(gated):
            fused = fused + g * weights[:, i].view(-1, 1, 1, 1)
        return self.old_bottleneck(fused)

    def extract_final_features(self, feats: List[torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor]:
        old_memory = self.forward_old(feats)
        if self.fusion is not None:
            spec_new = feats[-1]
            delta_old, feat_new = self.fusion(spec_new, old_memory.detach())
            enhanced_old = old_memory + delta_old
            final_features = torch.cat([enhanced_old, feat_new], dim=1) if feat_new is not None else enhanced_old
        else:
            final_features = old_memory
        flat_feat = F.normalize(self.pool(final_features).flatten(1), p=2, dim=1)
        return flat_feat, old_memory

    def forward(self, feats: List[torch.Tensor]) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        final_feat, old_memory = self.extract_final_features(feats)
        logits = self.classifier(final_feat)

        if self.use_weight_align and logits.size(1) > CLASSES_PER_TASK:
            with torch.no_grad():
                w_old = self.classifier.weight[:-CLASSES_PER_TASK]
                w_new = self.classifier.weight[-CLASSES_PER_TASK:]
                gamma = w_old.norm(dim=1).mean() / (w_new.norm(dim=1).mean() + 1e-8)
            logits[:, -CLASSES_PER_TASK:] *= gamma

        return logits, old_memory, final_feat


class KFN(nn.Module):
    def __init__(
        self,
        n_classes: int = 100,
        n_specs: int = 1,
        ch: int = 512,
        old_dim: int = 256,
        new_dim: int = 0,
        novelty_score: float = 0.0,
    ):
        super().__init__()
        self.specialists = nn.ModuleList([Specialist() for _ in range(n_specs)])
        self.global_model = GlobalModel(
            n_specs=n_specs,
            ch=ch,
            n_classes=n_classes,
            old_dim=old_dim,
            new_dim=new_dim,
            novelty_score=novelty_score,
            use_weight_align=True,
        )

    def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
        feats = [s(x) for s in self.specialists]
        return self.global_model(feats)

    def extract_backbone_features(self, x: torch.Tensor, spec_idx: int = 0) -> torch.Tensor:
        feat = self.specialists[spec_idx](x)
        return F.normalize(F.adaptive_avg_pool2d(feat, 1).flatten(1), p=2, dim=1)


def update_ema(student: nn.Module, teacher_ema: nn.Module, momentum: float = EMA_MOMENTUM) -> None:
    with torch.no_grad():
        for param_q, param_k in zip(student.parameters(), teacher_ema.parameters()):
            param_k.data.mul_(momentum).add_(param_q.data, alpha=1.0 - momentum)


def freeze_module(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad = False


def unfreeze_module(module: nn.Module) -> None:
    for p in module.parameters():
        p.requires_grad = True


def expand_kfn(
    old_model: KFN,
    trained_spec: Specialist,
    new_dim: int,
    novelty_score: float,
    n_classes: int,
) -> KFN:
    device = next(old_model.parameters()).device
    new_kfn = KFN(
        n_classes=n_classes,
        n_specs=len(old_model.specialists) + 1,
        ch=512,
        old_dim=old_model.global_model.old_dim,
        new_dim=new_dim,
        novelty_score=novelty_score,
    ).to(device)

    for i, spec in enumerate(old_model.specialists):
        new_kfn.specialists[i].load_state_dict(spec.state_dict())
    new_kfn.specialists[-1].load_state_dict(trained_spec.state_dict())

    old_g = old_model.global_model
    new_g = new_kfn.global_model

    for i in range(len(old_model.specialists)):
        new_g.old_proj[i].load_state_dict(old_g.old_proj[i].state_dict())
    new_g.old_gate.load_state_dict(old_g.old_gate.state_dict())
    new_g.old_bottleneck.load_state_dict(old_g.old_bottleneck.state_dict())
    new_g.spec_attention.load_state_dict(old_g.spec_attention.state_dict())

    with torch.no_grad():
        old_out, old_in = old_g.classifier.weight.shape
        new_out, new_in = new_g.classifier.weight.shape
        new_g.classifier.weight[:old_out, :old_in].copy_(old_g.classifier.weight)
        new_g.classifier.sigma.data.copy_(old_g.classifier.sigma.data)

        if new_out > old_out:
            nn.init.kaiming_normal_(new_g.classifier.weight[old_out:, :], nonlinearity="relu")
        if new_in > old_in:
            new_g.classifier.weight[:old_out, old_in:].zero_()

    return new_kfn


def compute_novelty(
    model: KFN,
    specialist: Specialist,
    loader: DataLoader,
    device: torch.device,
) -> Tuple[int, float]:
    model.eval()
    specialist.eval()
    scores = []

    with torch.no_grad():
        for x, _ in loader:
            x = x.to(device)
            f_new = specialist(x)
            f_new = F.normalize(F.adaptive_avg_pool2d(f_new, 1).flatten(1), p=2, dim=1)

            old_feats = [spec(x) for spec in model.specialists]
            old_memory = model.global_model.forward_old(old_feats)
            f_old = F.normalize(F.adaptive_avg_pool2d(old_memory, 1).flatten(1), p=2, dim=1)

            if f_old.size(1) != f_new.size(1):
                min_dim = min(f_old.size(1), f_new.size(1))
                f_old_cmp = f_old[:, :min_dim]
                f_new_cmp = f_new[:, :min_dim]
            else:
                f_old_cmp = f_old
                f_new_cmp = f_new

            novelty = 1.0 - torch.sum(f_new_cmp * f_old_cmp, dim=1)
            scores.append(novelty.mean().item())

    avg_score = float(np.mean(scores)) if scores else 0.0
    if avg_score > NOVELTY_MID:
        new_dim = 64
    elif avg_score > NOVELTY_LOW:
        new_dim = 32
    else:
        new_dim = 16
    return new_dim, avg_score


def evaluate(
    model: KFN,
    loader: DataLoader,
    device: torch.device,
) -> float:
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            logits, _, _ = model(x)
            pred = logits.argmax(dim=1)
            correct += (pred == y).sum().item()
            total += y.size(0)
    return 100.0 * correct / total if total > 0 else 0.0


def train_base_model(
    model: KFN,
    train_loader: DataLoader,
    device: torch.device,
    amp_ok: bool,
) -> None:
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_BASE)
    scaler = torch.amp.GradScaler("cuda") if amp_ok else None
    supcon = SupConLoss().to(device)

    for _ in range(EPOCHS_BASE):
        model.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            if amp_ok:
                with torch.amp.autocast("cuda"):
                    logits, _, feat = model(x)
                    loss = F.cross_entropy(logits, y) + 0.25 * supcon(feat, y)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                logits, _, feat = model(x)
                loss = F.cross_entropy(logits, y) + 0.25 * supcon(feat, y)
                loss.backward()
                opt.step()
        sched.step()


def train_specialist(
    specialist: Specialist,
    train_loader: DataLoader,
    task_id: int,
    device: torch.device,
    amp_ok: bool,
    criterion_supcon: nn.Module,
) -> None:
    head = nn.Linear(512, CLASSES_PER_TASK).to(device)
    opt = torch.optim.AdamW(list(specialist.parameters()) + list(head.parameters()), lr=1e-3, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_SPECIALIST)
    scaler = torch.amp.GradScaler("cuda") if amp_ok else None

    for _ in range(EPOCHS_SPECIALIST):
        specialist.train()
        head.train()
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            y_local = y - (task_id - 1) * CLASSES_PER_TASK
            opt.zero_grad()
            if amp_ok:
                with torch.amp.autocast("cuda"):
                    feat = specialist(x)
                    feat_flat = F.normalize(F.adaptive_avg_pool2d(feat, 1).flatten(1), p=2, dim=1)
                    logits = head(feat_flat)
                    loss_ce = F.cross_entropy(logits, y_local)
                    loss_supcon = criterion_supcon(feat_flat, y_local)
                    loss = loss_ce + 0.5 * loss_supcon
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                feat = specialist(x)
                feat_flat = F.normalize(F.adaptive_avg_pool2d(feat, 1).flatten(1), p=2, dim=1)
                logits = head(feat_flat)
                loss_ce = F.cross_entropy(logits, y_local)
                loss_supcon = criterion_supcon(feat_flat, y_local)
                loss = loss_ce + 0.5 * loss_supcon
                loss.backward()
                opt.step()
        sched.step()


def train_fusion_phase(
    model: KFN,
    teacher_prev: KFN,
    teacher_ema: KFN,
    train_loader: DataLoader,
    replay_loader: DataLoader,
    task_id: int,
    device: torch.device,
    amp_ok: bool,
    criterion_supcon: nn.Module,
) -> None:
    freeze_module(model)
    unfreeze_module(model.specialists[-1])
    unfreeze_module(model.global_model.classifier)
    unfreeze_module(model.global_model.old_gate)
    unfreeze_module(model.global_model.old_bottleneck)
    unfreeze_module(model.global_model.spec_attention)
    for proj in model.global_model.old_proj:
        unfreeze_module(proj)
    if model.global_model.fusion is not None:
        unfreeze_module(model.global_model.fusion)

    params = [
        {"params": model.specialists[-1].parameters(), "lr": 5e-4},
        {"params": model.global_model.old_proj.parameters(), "lr": 1e-4},
        {"params": model.global_model.old_gate.parameters(), "lr": 1e-4},
        {"params": model.global_model.old_bottleneck.parameters(), "lr": 1e-4},
        {"params": model.global_model.spec_attention.parameters(), "lr": 1e-4},
        {"params": model.global_model.classifier.parameters(), "lr": 1e-3},
    ]
    if model.global_model.fusion is not None:
        params.insert(0, {"params": model.global_model.fusion.parameters(), "lr": 1e-3})

    opt = torch.optim.AdamW(params, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_FUSION)
    scaler = torch.amp.GradScaler("cuda") if amp_ok else None

    for _ in range(EPOCHS_FUSION):
        model.train()
        replay_iter = iter(replay_loader)
        for x_curr, y_curr in train_loader:
            x_curr, y_curr = x_curr.to(device), y_curr.to(device)
            try:
                x_replay, y_replay = next(replay_iter)
            except StopIteration:
                replay_iter = iter(replay_loader)
                x_replay, y_replay = next(replay_iter)
            x_replay, y_replay = x_replay.to(device), y_replay.to(device)

            opt.zero_grad()

            if amp_ok:
                with torch.amp.autocast("cuda"):
                    logits_curr, _, _ = model(x_curr)
                    logits_replay, _, feat_replay = model(x_replay)

                    loss_curr = F.cross_entropy(logits_curr, y_curr)
                    loss_replay = F.cross_entropy(logits_replay, y_replay)

                    with torch.no_grad():
                        teacher_logits, _, teacher_feat_final = teacher_prev(x_replay)
                        _, _, ema_feat_final = teacher_ema(x_replay)

                    common_dim = teacher_logits.size(1)
                    student_logits = logits_replay[:, :common_dim]
                    loss_kd = F.kl_div(
                        F.log_softmax(student_logits / KD_TEMPERATURE, dim=1),
                        F.softmax(teacher_logits / KD_TEMPERATURE, dim=1),
                        reduction="batchmean",
                    ) * (KD_TEMPERATURE ** 2)

                    loss_feat = F.mse_loss(feat_replay[:, :teacher_feat_final.size(1)], teacher_feat_final)
                    loss_ema = F.mse_loss(feat_replay[:, :ema_feat_final.size(1)], ema_feat_final)

                    x_combined = torch.cat([x_curr, x_replay], dim=0)
                    y_combined = torch.cat([y_curr, y_replay], dim=0)
                    _, _, combined_feat = model(x_combined)
                    loss_supcon = criterion_supcon(combined_feat, y_combined)

                    with torch.no_grad():
                        _, _, teacher_feat_curr = teacher_prev(x_curr)
                    new_spec_feat = model.extract_backbone_features(x_curr, spec_idx=-1)
                    loss_cka = linear_cka(new_spec_feat, teacher_feat_curr)

                    adaptive_kd_weight = max(0.4, 1.2 - 0.08 * task_id)
                    loss = (
                        loss_curr
                        + loss_replay
                        + adaptive_kd_weight * loss_kd
                        + 0.25 * loss_feat
                        + 0.15 * loss_ema
                        + 0.4 * loss_supcon
                        + 0.1 * loss_cka
                    )

                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                logits_curr, _, _ = model(x_curr)
                logits_replay, _, feat_replay = model(x_replay)

                loss_curr = F.cross_entropy(logits_curr, y_curr)
                loss_replay = F.cross_entropy(logits_replay, y_replay)

                with torch.no_grad():
                    teacher_logits, _, teacher_feat_final = teacher_prev(x_replay)
                    _, _, ema_feat_final = teacher_ema(x_replay)

                common_dim = teacher_logits.size(1)
                student_logits = logits_replay[:, :common_dim]
                loss_kd = F.kl_div(
                    F.log_softmax(student_logits / KD_TEMPERATURE, dim=1),
                    F.softmax(teacher_logits / KD_TEMPERATURE, dim=1),
                    reduction="batchmean",
                ) * (KD_TEMPERATURE ** 2)

                loss_feat = F.mse_loss(feat_replay[:, :teacher_feat_final.size(1)], teacher_feat_final)
                loss_ema = F.mse_loss(feat_replay[:, :ema_feat_final.size(1)], ema_feat_final)

                x_combined = torch.cat([x_curr, x_replay], dim=0)
                y_combined = torch.cat([y_curr, y_replay], dim=0)
                _, _, combined_feat = model(x_combined)
                loss_supcon = criterion_supcon(combined_feat, y_combined)

                with torch.no_grad():
                    _, _, teacher_feat_curr = teacher_prev(x_curr)
                new_spec_feat = model.extract_backbone_features(x_curr, spec_idx=-1)
                loss_cka = linear_cka(new_spec_feat, teacher_feat_curr)

                adaptive_kd_weight = max(0.4, 1.2 - 0.08 * task_id)
                loss = (
                    loss_curr
                    + loss_replay
                    + adaptive_kd_weight * loss_kd
                    + 0.25 * loss_feat
                    + 0.15 * loss_ema
                    + 0.4 * loss_supcon
                    + 0.1 * loss_cka
                )

                loss.backward()
                opt.step()

            # 🔴 CORRECT PLACEMENT: Inside the batch loop
            update_ema(model, teacher_ema, EMA_MOMENTUM)

        sched.step()


def bias_correction_phase(
    model: KFN,
    replay_loader: DataLoader,
    device: torch.device,
    amp_ok: bool,
) -> None:
    freeze_module(model)
    unfreeze_module(model.global_model.classifier)
    if model.global_model.fusion is not None:
        unfreeze_module(model.global_model.fusion)

    params = [{"params": model.global_model.classifier.parameters(), "lr": 5e-4}]
    if model.global_model.fusion is not None:
        params.append({"params": model.global_model.fusion.parameters(), "lr": 1e-4})

    opt = torch.optim.AdamW(params, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS_BIAS)
    scaler = torch.amp.GradScaler("cuda") if amp_ok else None

    for _ in range(EPOCHS_BIAS):
        model.train()
        for x_replay, y_replay in replay_loader:
            x_replay, y_replay = x_replay.to(device), y_replay.to(device)
            opt.zero_grad()
            if amp_ok:
                with torch.amp.autocast("cuda"):
                    logits, _, _ = model(x_replay)
                    loss = F.cross_entropy(logits, y_replay)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                logits, _, _ = model(x_replay)
                loss = F.cross_entropy(logits, y_replay)
                loss.backward()
                opt.step()
        sched.step()


def train_kfn_split_cifar100(
    seed: int = 0,
    quick_run: bool = QUICK_RUN,
    save_path: str = SAVE_PATH,
) -> Dict:
    set_deterministic(seed)
    device = get_device(FORCE_CPU)
    amp_ok = USE_AMP and device.type == "cuda"
    criterion_supcon = SupConLoss().to(device)

    print(f"🚀 Training Improved KFN on Split CIFAR-100 | Seed: {seed} | Device: {device}")

    stats = ((0.5071, 0.4865, 0.4409), (0.2673, 0.2564, 0.2762))
    t_train = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
        transforms.ToTensor(),
        transforms.Normalize(*stats),
    ])
    t_test = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(*stats),
    ])

    root = "./data"
    train_ds = torchvision.datasets.CIFAR100(root=root, train=True, download=True, transform=t_train)
    test_ds = torchvision.datasets.CIFAR100(root=root, train=False, download=True, transform=t_test)

    replay_buffer = DynamicReplayBuffer(REPLAY_BUFFER_CAPACITY)
    accuracy_matrix = np.zeros((N_TASKS, N_TASKS))
    param_counts = []
    results = {
        "accuracy_matrix": accuracy_matrix.tolist(),
        "average_accuracy": [],
        "param_counts": param_counts,
        "tasks": [],
    }

    print("\n🔹 Task 1: Base Initialization (Classes 0-9)")
    train_loader, test_loader = get_task_loaders(train_ds, test_ds, task_id=1)
    model = KFN(n_classes=CLASSES_PER_TASK, n_specs=1).to(device)
    train_base_model(model, train_loader, device, amp_ok)

    acc = evaluate(model, test_loader, device)
    accuracy_matrix[0, 0] = acc
    print(f"✅ Task 1 Accuracy: {acc:.2f}%")

    replay_buffer.add_exemplars_from_dataset(
        dataset=train_ds,
        classes=get_task_classes(1),
        feature_extractor=model.specialists[0],
        device=device,
    )

    teacher_prev = copy.deepcopy(model).eval()
    teacher_ema = copy.deepcopy(model).eval()
    freeze_module(teacher_prev)
    freeze_module(teacher_ema)

    param_counts.append(sum(p.numel() for p in model.parameters()))
    results["average_accuracy"].append(float(acc))
    results["param_counts"] = param_counts
    results["tasks"].append({
        "task_id": 1,
        "classes": get_task_classes(1),
        "novelty_score": 0.0,
        "expansion_dim": 0,
        "param_count": param_counts[-1],
    })
    results["accuracy_matrix"] = accuracy_matrix.tolist()
    save_progress(results, save_path)

    for task_id in range(2, N_TASKS + 1):
        class_start = CLASSES_PER_TASK * (task_id - 1)
        class_end = CLASSES_PER_TASK * task_id - 1
        print(f"\n🔹 Task {task_id}: Incremental Learning (Classes {class_start}-{class_end})")
        current_classes = get_task_classes(task_id)
        train_loader, test_loader = get_task_loaders(train_ds, test_ds, task_id=task_id)

        print("  🔸 Phase 2: Specialist Training")
        specialist = Specialist().to(device)
        train_specialist(specialist, train_loader, task_id, device, amp_ok, criterion_supcon)

        print("  🔸 Phase 3A: Novelty Detection & Expansion")
        new_dim, novelty_score = compute_novelty(teacher_prev, specialist, test_loader, device)
        print(f"    Novelty Score: {novelty_score:.4f} | Expansion Dim: {new_dim}")

        print("  🔸 Phase 3B: Structural Fusion")
        model = expand_kfn(
            teacher_prev,
            specialist,
            new_dim,
            novelty_score,
            n_classes=CLASSES_PER_TASK * task_id,
        ).to(device)

        teacher_ema = copy.deepcopy(model).eval()
        freeze_module(teacher_ema)

        replay_loader = replay_buffer.get_loader()
        if replay_loader is None:
            raise RuntimeError("Replay buffer is empty before incremental fusion training.")

        train_fusion_phase(
            model,
            teacher_prev,
            teacher_ema,
            train_loader,
            replay_loader,
            task_id,
            device,
            amp_ok,
            criterion_supcon,
        )

        replay_buffer.add_exemplars_from_dataset(
            dataset=train_ds,
            classes=current_classes,
            feature_extractor=model.specialists[-1],
            device=device,
        )

        print("  🔸 Phase 4: Bias Correction")
        replay_loader = replay_buffer.get_loader(shuffle=True)
        if replay_loader is None:
            raise RuntimeError("Replay buffer is empty before bias correction.")
        bias_correction_phase(model, replay_loader, device, amp_ok)

        print("  🔸 Evaluation")
        for t in range(1, task_id + 1):
            _, test_loader_t = get_task_loaders(train_ds, test_ds, task_id=t)
            task_acc = evaluate(model, test_loader_t, device)
            accuracy_matrix[task_id - 1, t - 1] = task_acc
            print(f"    Task {t} Accuracy: {task_acc:.2f}%")

        teacher_prev = copy.deepcopy(model).eval()
        freeze_module(teacher_prev)

        param_counts.append(sum(p.numel() for p in model.parameters()))
        avg_acc = float(np.mean(accuracy_matrix[task_id - 1, :task_id]))
        results["average_accuracy"].append(avg_acc)
        results["accuracy_matrix"] = accuracy_matrix.tolist()
        results["param_counts"] = param_counts
        results["tasks"].append({
            "task_id": task_id,
            "classes": current_classes,
            "novelty_score": novelty_score,
            "expansion_dim": new_dim,
            "param_count": param_counts[-1],
        })
        save_progress(results, save_path)

    print("\n🎉 Training Complete!")
    print(f"📊 Final Accuracy Matrix:\n{np.array(results['accuracy_matrix'])}")
    print(f"📈 Average Accuracy: {results['average_accuracy'][-1]:.2f}%")
    print(f"📏 Parameter Count: {results['param_counts'][-1]:,}")
    return results


if __name__ == "__main__":
    train_kfn_split_cifar100(seed=0, quick_run=QUICK_RUN)

In [ ]:
'''
ubuntu@gt-ubuntu24-04-cmd-v3-2-120gb-100m:~$ python3 kfn++.py
🚀 Training Improved KFN on Split CIFAR-100 | Seed: 0 | Device: cuda

🔹 Task 1: Base Initialization (Classes 0-9)
✅ Task 1 Accuracy: 93.00%

🔹 Task 2: Incremental Learning (Classes 10-19)
  🔸 Phase 2: Specialist Training
  🔸 Phase 3A: Novelty Detection & Expansion
    Novelty Score: 0.5686 | Expansion Dim: 64
  🔸 Phase 3B: Structural Fusion
  🔸 Phase 4: Bias Correction
  🔸 Evaluation
    Task 1 Accuracy: 76.90%
    Task 2 Accuracy: 84.60%

🔹 Task 3: Incremental Learning (Classes 20-29)
  🔸 Phase 2: Specialist Training
  🔸 Phase 3A: Novelty Detection & Expansion
    Novelty Score: 0.6601 | Expansion Dim: 64
  🔸 Phase 3B: Structural Fusion
  🔸 Phase 4: Bias Correction
  🔸 Evaluation
    Task 1 Accuracy: 72.30%
    Task 2 Accuracy: 72.70%
    Task 3 Accuracy: 81.10%

🔹 Task 4: Incremental Learning (Classes 30-39)
  🔸 Phase 2: Specialist Training
  🔸 Phase 3A: Novelty Detection & Expansion
    Novelty Score: 0.7092 | Expansion Dim: 64
  🔸 Phase 3B: Structural Fusion

  🔸 Phase 4: Bias Correction
  🔸 Evaluation
    Task 1 Accuracy: 67.00%
    Task 2 Accuracy: 65.10%
    Task 3 Accuracy: 75.70%
    Task 4 Accuracy: 77.00%

🔹 Task 5: Incremental Learning (Classes 40-49)
  🔸 Phase 2: Specialist Training
  🔸 Phase 3A: Novelty Detection & Expansion
    Novelty Score: 0.7124 | Expansion Dim: 64
  🔸 Phase 3B: Structural Fusion

  🔸 Phase 4: Bias Correction
  🔸 Evaluation
    Task 1 Accuracy: 64.00%
    Task 2 Accuracy: 61.00%
    Task 3 Accuracy: 72.20%
    Task 4 Accuracy: 73.10%
    Task 5 Accuracy: 74.90%

🔹 Task 6: Incremental Learning (Classes 50-59)
  🔸 Phase 2: Specialist Training
  🔸 Phase 3A: Novelty Detection & Expansion
    Novelty Score: 0.7008 | Expansion Dim: 64
  🔸 Phase 3B: Structural Fusion


  🔸 Phase 4: Bias Correction
  🔸 Evaluation
    Task 1 Accuracy: 61.20%
    Task 2 Accuracy: 58.10%
    Task 3 Accuracy: 69.80%
    Task 4 Accuracy: 70.20%
    Task 5 Accuracy: 70.90%
    Task 6 Accuracy: 73.70%

🔹 Task 7: Incremental Learning (Classes 60-69)
  🔸 Phase 2: Specialist Training
  🔸 Phase 3A: Novelty Detection & Expansion
    Novelty Score: 0.7239 | Expansion Dim: 64
  🔸 Phase 3B: Structural Fusion


  🔸 Phase 4: Bias Correction

  🔸 Evaluation
    Task 1 Accuracy: 60.20%
    Task 2 Accuracy: 57.00%
    Task 3 Accuracy: 68.90%
    Task 4 Accuracy: 67.20%
    Task 5 Accuracy: 67.50%
    Task 6 Accuracy: 74.50%
    Task 7 Accuracy: 68.00%

🔹 Task 8: Incremental Learning (Classes 70-79)
  🔸 Phase 2: Specialist Training
  🔸 Phase 3A: Novelty Detection & Expansion
    Novelty Score: 0.7105 | Expansion Dim: 64
  🔸 Phase 3B: Structural Fusion


  🔸 Phase 4: Bias Correction
  🔸 Evaluation
    Task 1 Accuracy: 58.60%
    Task 2 Accuracy: 53.80%
    Task 3 Accuracy: 67.60%
    Task 4 Accuracy: 66.60%
    Task 5 Accuracy: 66.00%
    Task 6 Accuracy: 73.60%
    Task 7 Accuracy: 68.00%
    Task 8 Accuracy: 62.40%

🔹 Task 9: Incremental Learning (Classes 80-89)
  🔸 Phase 2: Specialist Training
  🔸 Phase 3A: Novelty Detection & Expansion
    Novelty Score: 0.7123 | Expansion Dim: 64
  🔸 Phase 3B: Structural Fusion
  🔸 Phase 4: Bias Correction
  🔸 Evaluation
    Task 1 Accuracy: 57.90%
    Task 2 Accuracy: 53.70%
    Task 3 Accuracy: 66.40%
    Task 4 Accuracy: 64.70%
    Task 5 Accuracy: 65.50%
    Task 6 Accuracy: 72.40%
    Task 7 Accuracy: 65.30%
    Task 8 Accuracy: 64.30%
    Task 9 Accuracy: 65.40%

🔹 Task 10: Incremental Learning (Classes 90-99)
  🔸 Phase 2: Specialist Training
  🔸 Phase 3A: Novelty Detection & Expansion
    Novelty Score: 0.7146 | Expansion Dim: 64
  🔸 Phase 3B: Structural Fusion

  🔸 Phase 4: Bias Correction


  🔸 Evaluation
    Task 1 Accuracy: 56.00%
    Task 2 Accuracy: 51.40%
    Task 3 Accuracy: 64.80%
    Task 4 Accuracy: 61.40%
    Task 5 Accuracy: 62.00%
    Task 6 Accuracy: 70.40%
    Task 7 Accuracy: 62.00%
    Task 8 Accuracy: 63.00%
    Task 9 Accuracy: 69.30%
    Task 10 Accuracy: 60.30%

🎉 Training Complete!
📊 Final Accuracy Matrix:
[[93.   0.   0.   0.   0.   0.   0.   0.   0.   0. ]
 [76.9 84.6  0.   0.   0.   0.   0.   0.   0.   0. ]
 [72.3 72.7 81.1  0.   0.   0.   0.   0.   0.   0. ]
 [67.  65.1 75.7 77.   0.   0.   0.   0.   0.   0. ]
 [64.  61.  72.2 73.1 74.9  0.   0.   0.   0.   0. ]
 [61.2 58.1 69.8 70.2 70.9 73.7  0.   0.   0.   0. ]
 [60.2 57.  68.9 67.2 67.5 74.5 68.   0.   0.   0. ]
 [58.6 53.8 67.6 66.6 66.  73.6 68.  62.4  0.   0. ]
 [57.9 53.7 66.4 64.7 65.5 72.4 65.3 64.3 65.4  0. ]
 [56.  51.4 64.8 61.4 62.  70.4 62.  63.  69.3 60.3]]
📈 Average Accuracy: 62.06%
📏 Parameter Count: 16,429,763

'''